In [0]:
import logging
from pyspark.sql.functions import current_timestamp, lit

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
log = logging.getLogger(__name__)

RAW_SOCIAL_PATH    = "abfss://raw@cryptodl.dfs.core.windows.net/JSON_Streaming_Data_Source_2/"
BRONZE_OUTPUT_PATH = "abfss://bronzelayer@cryptodl.dfs.core.windows.net/JSON_Streaming_Data_Source_2/datafiles/"
SCHEMA_PATH        = "abfss://bronzelayer@cryptodl.dfs.core.windows.net/JSON_Streaming_Data_Source_2/_schema/"
CHECKPOINT_PATH = "abfss://bronzelayer@cryptodl.dfs.core.windows.net/JSON_Streaming_Data_Source_2/_checkpoint/"


log.info("Starting Source 2 Bronze Load")
log.info(f"Source : {RAW_SOCIAL_PATH}")
log.info(f"Target : {BRONZE_OUTPUT_PATH}")

social_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("cloudFiles.schemaLocation", SCHEMA_PATH)
         .load(RAW_SOCIAL_PATH)
)

bronze_social = (
    social_stream
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("data_source",         lit("social_media_stream"))
)

query = (
    bronze_social.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start(BRONZE_OUTPUT_PATH)
)

query.awaitTermination()
log.info(f"Stream complete — rows processed: {query.lastProgress.get('numInputRows', 'N/A')}")


In [0]:
log.info("Reading Bronze Delta for verification...")
bronze_df = spark.read.format("delta").load(BRONZE_OUTPUT_PATH)

total_rows    = bronze_df.count()
distinct_posts = bronze_df.select("post_id").distinct().count()
duplicates     = total_rows - distinct_posts

log.info(f"Total rows       : {total_rows:,}")
log.info(f"Distinct post IDs: {distinct_posts:,}")

if duplicates == 0:
    log.info("No duplicates found ")
else:
    log.warning(f"Duplicates found: {duplicates}")

display(bronze_df.limit(20))